# 04 · Tuning methods compared at an equal evaluation budget

Is the three-stage search (Silverback → Dolphin → pruning) better than simpler ways of tuning a decision tree? All methods receive the same budget `B`
of evaluations of the same cross-validated fitness, on the same rows, and are compared after the same refit.

**Protocol**
* **Common re-scoring.** Silverback adds a novelty bonus to its fitness; every method's winner is therefore re-scored by the same plain evaluator (`rescored_fitness`), and convergence curves use the bonus-free CV-MCC.
* **Common search space.** Random search and Optuna search the union of the staged methods' spaces, so they can reach every configuration the staged methods can.
* **Common refit.** All winners are refitted on the same rows (`OPT_CMP_REFIT_LIMIT`, 0 = all training rows) and scored on the same validation and test rows.
* **Repeats and statistics.** Repeats are paired by seed; differences are reported with paired bootstrap 95 % CIs, Wilcoxon tests, win rates and mean ranks.
* **Simple baselines.** A default tree (no search) and a pruning grid over `max_depth` × `min_samples_leaf` × `ccp_alpha`.
* **Deployability.** A run is deployable when its tree has at most `NODE_CAP` nodes and its MCC is within `MCC_TOL` of the best run of that repeat.
* **Exported size.** 11 B per node plus the preprocessing arrays (legacy layout; the exact export uses 10 B per node); the 24 B/node search proxy is kept as an upper bound.
* **Resumable.** A progress line and a partial save are written after every run.

Methods: `Full (S→D→P)` (47/32/21 % of the budget) · `Silverback only` · `Silverback→Dolphin` · `Silverback→Prune` · `Random search` · `Optuna TPE` · `Pruning-grid search` · `Default DT (no search)`.

Methods are ranked on **validation** metrics; test metrics are reported but not used for ranking.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default
import copy, random as _random, time, datetime, itertools, shutil
from scipy import stats as _st
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False
print("Dataset:", ACTIVE_DATASET, "| budget B =", OPT_CMP_BUDGET, "| repeats =", OPT_CMP_REPEATS, "| optuna:", HAS_OPTUNA)
print(f"search sample {OPT_CMP_SAMPLE_LIMIT:,} rows | refit {'ALL training rows' if not OPT_CMP_REFIT_LIMIT else format(OPT_CMP_REFIT_LIMIT, ',') + ' rows'} | scoring {'FULL validation/test splits' if not OPT_CMP_EVAL_LIMIT else format(OPT_CMP_EVAL_LIMIT, ',') + ' rows'}")

## Data, split, fixed evaluation sets

In [ ]:
artifact, metadata = load_artifact_and_metadata()
X, y, le_check, numeric_cols_check = load_publishability_data()
X_train, X_val, X_test, y_train, y_val, y_test, X_final, y_final = same_split(X, y)
class_names = list(le_check.classes_)
BEST_K = int(metadata["best_k"])
rare_classes, extremely_rare_classes = infer_rare_classes(y_train)

# Validation / test scoring sets: ONE fixed stratified subset (identical for every method and repeat)
REFIT_N = OPT_CMP_REFIT_LIMIT or len(X_train)          # 0 -> every training row (paper scale)
EVAL_N = OPT_CMP_EVAL_LIMIT or max(len(X_val), len(X_test))
Xv, yv = stratified_sample(X_val, y_val, EVAL_N, SEED)
Xt, yt = stratified_sample(X_test, y_test, EVAL_N, SEED)

# ---- deployability criteria (environment variables override) ----------------------
saved_nodes = int(artifact["pipeline"].named_steps["clf"].tree_.node_count)
NODE_CAP = int(os.environ.get("OPT_CMP_NODE_CAP", max(1000, 2 * saved_nodes)))   # "fits the ESP32 budget"
MCC_TOL = float(os.environ.get("OPT_CMP_MCC_TOL", 0.01))                         # "within 0.01 MCC of the best run"
def exported_kb(nodes, k=BEST_K):          # real C layout: 3x int16 + float32 + uint8 = 11 B / node, + 13 B / selected feature
    return (11 * nodes + 13 * k) / 1024

B = OPT_CMP_BUDGET
S_full, D_full = round(.47 * B), round(.32 * B); P_full = B - S_full - D_full
print(f"K = {BEST_K} | saved (full-mode) model has {saved_nodes:,} nodes -> NODE_CAP = {NODE_CAP:,} nodes (~{exported_kb(NODE_CAP):.0f} KB exported) | MCC_TOL = {MCC_TOL}")
print(f"Full-method budget split: Silverback {S_full} / Dolphin {D_full} / Prune {P_full}  (total {B})")
print(f"validation scoring rows: {len(Xv):,} | test scoring rows: {len(Xt):,}")

## The optimisers
All of them call the *same* `evaluate_candidate` (3-fold CV inside the leakage-safe pipeline). Each returns `(final_params, list_of_per_evaluation_metrics)`.

* **Union space:** random search and Optuna sample from `DT_SEARCH_SPACE ∪ PRUNE_SEARCH_SPACE` (value lists merged per key) — so they can reach everything the staged methods can.
* **Pruning-grid search** (the *simple tuning* baseline): only `max_depth` × `min_samples_leaf` × `ccp_alpha` are searched over the full grid (11 × 7 × 7 = 539 points), everything else is the scikit-learn default. It gets the **same budget B** as every other method, i.e. a random subset of B grid points when B < 539 (all points otherwise).

In [ ]:
# ---- union search space -------------------------------------------------------------
UNION_SPACE = {}
for key in set(DT_SEARCH_SPACE) | set(PRUNE_SEARCH_SPACE):
    vals = []
    for v in list(DT_SEARCH_SPACE.get(key, [])) + list(PRUNE_SEARCH_SPACE.get(key, [])):
        if not any((v is None and u is None) or (v is not None and u is not None and v == u) for u in vals):
            vals.append(v)
    UNION_SPACE[key] = vals

# Simple-tuning baseline: the three classic regularisation knobs of a decision tree, everything else at the scikit-learn default.
# (min_samples_leaf is the knob that shrinks the NF-ToN-IoT-v2 tree from 616,073 to 7,073 nodes.)
PRUNE_GRID = [{"max_depth": d, "min_samples_leaf": l, "ccp_alpha": a}
              for d, l, a in itertools.product(PRUNE_SEARCH_SPACE["max_depth"], PRUNE_SEARCH_SPACE["min_samples_leaf"], PRUNE_SEARCH_SPACE["ccp_alpha"])]
print(f"pruning grid: {len(PRUNE_SEARCH_SPACE['max_depth'])} depths x {len(PRUNE_SEARCH_SPACE['min_samples_leaf'])} leaf sizes x {len(PRUNE_SEARCH_SPACE['ccp_alpha'])} ccp_alpha = {len(PRUNE_GRID)} points; "
      f"with B = {B} evaluations the baseline visits {'a random subset of' if B < len(PRUNE_GRID) else 'all'} them")

def _ev(params, Xs, ys, seed):
    return evaluate_candidate(params, BEST_K, Xs, ys, seed, rare_classes, extremely_rare_classes)

def run_random(Xs, ys, seed):
    rng = _random.Random(seed); hist, best = [], None
    for _ in range(B):
        c = _ev({k: rng.choice(v) for k, v in UNION_SPACE.items()}, Xs, ys, seed)
        hist.append(c["metrics"])
        if best is None or c["metrics"]["fitness"] > best["metrics"]["fitness"]: best = c
    return best["params"], hist

def run_tpe(Xs, ys, seed):
    hist, store = [], {}
    def objective(trial):
        params = {k: trial.suggest_categorical(k, [("__none__" if v is None else v) for v in vals]) for k, vals in UNION_SPACE.items()}
        params = {k: (None if v == "__none__" else v) for k, v in params.items()}
        c = _ev(params, Xs, ys, seed); hist.append(c["metrics"]); store[trial.number] = c["params"]
        return c["metrics"]["fitness"]
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    st.optimize(objective, n_trials=B)
    return store[st.best_trial.number], hist

def run_prune_grid(Xs, ys, seed):
    rng = _random.Random(seed); pts = PRUNE_GRID[:]
    if len(pts) > B: pts = rng.sample(pts, B)
    hist, best = [], None
    for p in pts:
        c = _ev(dict(p), Xs, ys, seed); hist.append(c["metrics"])
        if best is None or c["metrics"]["fitness"] > best["metrics"]["fitness"]: best = c
    return best["params"], hist

def run_staged(Xs, ys, seed, s, d, p):
    hist, params = [], None
    if s:
        best, h = run_silverback_enhanced(Xs, ys, BEST_K, s, seed, rare_classes, extremely_rare_classes, verbose=False); hist += list(h); params = best["params"]
    if d:
        best, h = run_dolphin_enhanced(params, Xs, ys, BEST_K, d, seed, rare_classes, extremely_rare_classes, verbose=False); hist += list(h); params = best["params"]
    if p:
        best, h = run_prune_silverback(params, Xs, ys, BEST_K, p, seed, rare_classes, extremely_rare_classes); hist += list(h); params = best["params"]
    return params, hist

METHODS = {
    "Full (S→D→P)":         lambda Xs, ys, sd: run_staged(Xs, ys, sd, S_full, D_full, P_full),
    "Silverback only":      lambda Xs, ys, sd: run_staged(Xs, ys, sd, B, 0, 0),
    "Silverback→Dolphin":   lambda Xs, ys, sd: run_staged(Xs, ys, sd, B - D_full, D_full, 0),
    "Silverback→Prune":     lambda Xs, ys, sd: run_staged(Xs, ys, sd, B - P_full, 0, P_full),
    "Random search":        lambda Xs, ys, sd: run_random(Xs, ys, sd),
    "Pruning-grid search":  lambda Xs, ys, sd: run_prune_grid(Xs, ys, sd),
    "Default DT (no search)": lambda Xs, ys, sd: ({}, []),
}
if HAS_OPTUNA:
    METHODS["Optuna TPE"] = lambda Xs, ys, sd: run_tpe(Xs, ys, sd)
print("methods:", list(METHODS))

## Optional: restricted-space variants (`OPT_CMP_LADDER=1`)
The original fitness is a weighted sum of eight metrics in which rare-class F1 and minority recall carry the largest weights, its search space contains options that
hurt on these data (`splitter=random`, `max_features<1`, `class_weight=balanced`, `min_weight_fraction_leaf` up to 0.08), and its novelty bonus is unbounded.
Each row below removes one of these cumulatively:

| Method | Change relative to the previous row |
|---|---|
| `Full + restricted space` | search only `criterion, max_depth, min_samples_split, min_samples_leaf (up to 64), ccp_alpha` |
| `Full + restricted space + MCC objective` | fitness = CV MCC − small size penalty |
| `Full + restricted + MCC + no novelty bonus` | novelty bonus removed |
| `Halving-Silverback (aligned)` | same space and objective with successive halving: about 1.9× more candidates screened on 25 % / 50 % of the rows; compute-matched to the budget (compare `search_seconds`) |
| `Random search (aligned)` / `Pruning-grid (aligned)` | the baselines under the same space and objective |

> Variants should be designed on one dataset (here NF-UNSW-NB15-v3) and confirmed unchanged on another (NF-ToN-IoT-v2); choosing the best variant on the reported data inflates the result.

In [ ]:
import contextlib
LADDER = os.environ.get("OPT_CMP_LADDER", "0") == "1"
ALIGNED_SPACE = {"criterion": ["gini", "entropy"], "max_depth": [8, 10, 12, 14, 16, 18, 20, 24, 28, 32, None],
                 "min_samples_split": [2, 5, 10, 20, 40], "min_samples_leaf": [1, 2, 4, 8, 16, 32, 64], "ccp_alpha": [0.0, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]}
SIZE_BUDGET_KB = 800.0
def fitness_mcc(cv):
    kb = cv["node_count"] * 11 / 1024                                   # exported size, 11 B / node
    return float(cv["cv_mcc_mean"] - 0.02 * kb / SIZE_BUDGET_KB - max(0.0, kb - SIZE_BUDGET_KB) / SIZE_BUDGET_KB)

@contextlib.contextmanager
def aligned(space=True, objective=True, no_novelty=True):
    """Temporarily switch the framework to the restricted space / MCC objective / no novelty bonus. Always restored."""
    import framework as fw
    old = (fw.DT_SEARCH_SPACE, fw.compute_fitness, fw.EnhancedSilverback.novelty_bonus)
    try:
        if space: fw.DT_SEARCH_SPACE = ALIGNED_SPACE
        if objective: fw.compute_fitness = fitness_mcc
        if no_novelty: fw.EnhancedSilverback.novelty_bonus = lambda self, params: 0.0
        yield
    finally:
        fw.DT_SEARCH_SPACE, fw.compute_fitness, fw.EnhancedSilverback.novelty_bonus = old

def run_random_aligned(Xs, ys, seed):
    rng = _random.Random(seed); hist, best = [], None
    with aligned(False, True, True):
        for _ in range(B):
            c = _ev({k: rng.choice(v) for k, v in ALIGNED_SPACE.items()}, Xs, ys, seed); hist.append(c["metrics"])
            if best is None or c["metrics"]["fitness"] > best["metrics"]["fitness"]: best = c
    return best["params"], hist

def run_grid_aligned(Xs, ys, seed):
    rng = _random.Random(seed); pts = PRUNE_GRID[:]
    if len(pts) > B: pts = rng.sample(pts, B)
    hist, best = [], None
    with aligned(False, True, True):
        for p_ in pts:
            c = _ev(dict(p_), Xs, ys, seed); hist.append(c["metrics"])
            if best is None or c["metrics"]["fitness"] > best["metrics"]["fitness"]: best = c
    return best["params"], hist

def _children(rng, survivors, n):
    out = []
    for _ in range(n):
        if len(survivors) > 1 and rng.random() < 0.4:                    # Silverback-style crossover
            a, b = rng.sample(survivors, 2); out.append({k: (a[k] if rng.random() < .5 else b[k]) for k in ALIGNED_SPACE})
        else:                                                           # Dolphin-style local move: neighbouring value of one or two knobs
            c = dict(rng.choice(survivors))
            for k in rng.sample(list(ALIGNED_SPACE), rng.choice([1, 2])):
                vals = ALIGNED_SPACE[k]; i = vals.index(c[k]); c[k] = vals[max(0, min(len(vals) - 1, i + rng.choice([-1, 1])))]
            out.append(c)
    return out

def run_halving(Xs, ys, seed):
    """Successive halving with Silverback/Dolphin-style proposals; compute-matched to B full-sample evaluations."""
    rng = _random.Random(seed + 4242); fr, keep = [0.25, 0.5, 1.0], 1 / 3
    n0 = max(6, int(round(B / (fr[0] + fr[1] * keep + fr[2] * keep * keep))))
    subs = [stratified_sample(Xs, ys, max(int(f * len(Xs)), min(len(Xs), 3000)), seed + i) for i, f in enumerate(fr)]
    hist = []
    with aligned(True, True, True):
        pool = [{k: rng.choice(v) for k, v in ALIGNED_SPACE.items()} for _ in range(n0)]
        for r, (Xr, yr) in enumerate(subs):
            scored = []
            for p_ in pool:
                c = _ev(p_, Xr, yr, seed); scored.append((c["metrics"]["fitness"], p_)); hist.append(c["metrics"])
            scored.sort(key=lambda t: -t[0])
            if r < len(subs) - 1:
                target = max(2, int(np.ceil(n0 * keep ** (r + 1)))); surv = [p_ for _, p_ in scored[:max(2, target // 2)]]
                pool = surv + _children(rng, surv, max(0, target - len(surv)))
        best = scored[0][1]
    return best, hist

if LADDER:
    def staged_a(s, d, p, **kw):
        def f(Xs, ys, sd):
            with aligned(**kw): return run_staged(Xs, ys, sd, s, d, p)
        return f
    METHODS["Full + restricted space"] = staged_a(S_full, D_full, P_full, space=True, objective=False, no_novelty=False)
    METHODS["Full + restricted space + MCC objective"] = staged_a(S_full, D_full, P_full, space=True, objective=True, no_novelty=False)
    METHODS["Full + restricted + MCC + no novelty bonus"] = staged_a(S_full, D_full, P_full, space=True, objective=True, no_novelty=True)
    METHODS["Halving-Silverback (aligned)"] = lambda Xs, ys, sd: run_halving(Xs, ys, sd)
    METHODS["Random search (aligned)"] = lambda Xs, ys, sd: run_random_aligned(Xs, ys, sd)
    METHODS["Pruning-grid (aligned)"] = lambda Xs, ys, sd: run_grid_aligned(Xs, ys, sd)
print("ladder:", LADDER, "| methods:", list(METHODS))

## Run *(slow, resumable)*
After every (repeat, method) the raw results are saved and a line is written to `progress.txt`. An interrupted run continues where it stopped.
If the budget, sample sizes or K change, existing files are moved to `previous_<timestamp>/` and the run starts again. Increasing the number of repeats extends an existing result.

In [ ]:
OUT = Path("optimizer_comparison") / ACTIVE_DATASET; OUT.mkdir(parents=True, exist_ok=True)
cfg = {"protocol": "v2", "budget": B, "k": BEST_K, "search_rows": OPT_CMP_SAMPLE_LIMIT, "refit_rows": REFIT_N,
       "eval_rows": EVAL_N, "methods": list(METHODS), "ladder": LADDER, "prune_grid": "max_depth x min_samples_leaf x ccp_alpha", "seed_base": SEED + 700, "node_cap": NODE_CAP, "mcc_tol": MCC_TOL}
cfg_file, raw_file, cur_file, prog_file = OUT / "run_config.json", OUT / "optimizer_comparison_raw.csv", OUT / "optimizer_comparison_curves.csv", OUT / "progress.txt"

old_cfg = json.loads(cfg_file.read_text()) if cfg_file.exists() else None
if old_cfg != cfg and (raw_file.exists() or old_cfg is not None):
    arch = OUT / ("previous_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")); arch.mkdir()
    for f in OUT.glob("*"):
        if f.is_file(): shutil.move(str(f), arch / f.name)
    print("settings changed -> earlier results archived to", arch.name); old_cfg = None
cfg_file.write_text(json.dumps(cfg, indent=2, default=str))

rows = pd.read_csv(raw_file).to_dict("records") if raw_file.exists() else []
curves = pd.read_csv(cur_file).to_dict("records") if cur_file.exists() else []
done = {(int(r["repeat"]), r["method"]) for r in rows}
print(f"resuming with {len(done)} finished runs" if done else "fresh start")

def log(msg):
    print(msg, flush=True)
    with open(prog_file, "a", encoding="utf-8") as f: f.write(datetime.datetime.now().strftime("%H:%M:%S ") + msg + "\n")

t_start, n_new = time.perf_counter(), 0
total_runs = OPT_CMP_REPEATS * len(METHODS)
for rep in range(OPT_CMP_REPEATS):
    todo = [m for m in METHODS if (rep, m) not in done]
    if not todo: continue
    seed = SEED + 700 + rep
    Xs, ys = stratified_sample(X_train, y_train, OPT_CMP_SAMPLE_LIMIT, seed)           # search sample
    Xr, yr = stratified_sample(X_train, y_train, REFIT_N, seed)                       # refit rows (same for all methods; all training rows at paper scale)
    Xq, yq = stratified_sample(X_train, y_train, OPT_CMP_SAMPLE_LIMIT, seed + 999)     # FRESH sample: plain re-scoring of every winner
    for name in todo:
        t0 = time.perf_counter(); params, hist = METHODS[name](Xs, ys, seed); search_s = time.perf_counter() - t0
        rs = _ev(params, Xq, yq, seed + 999)["metrics"]                                 # same plain evaluator, no novelty bonus
        t1 = time.perf_counter(); pipe = build_pipeline(params, BEST_K, seed); pipe.fit(Xr, yr); refit_s = time.perf_counter() - t1
        mv = compute_enhanced_metrics(yv, pipe.predict(Xv), rare_classes, extremely_rare_classes)
        mt = compute_enhanced_metrics(yt, pipe.predict(Xt), rare_classes, extremely_rare_classes)
        tree = pipe.named_steps["clf"]; nodes = int(tree.tree_.node_count)
        rows.append({"method": name, "repeat": rep, "seed": seed, "n_evals": len(hist), "search_rows": len(Xs), "refit_rows": len(Xr),
                     "search_seconds": search_s, "refit_seconds": refit_s,
                     "rescored_fitness": rs["fitness"], "rescored_cv_mcc": rs["cv_mcc_mean"],
                     "val_mcc": mv["mcc"], "val_macro_f1": mv["macro_f1"], "val_accuracy": mv["accuracy"],
                     "val_minority_recall": mv["minority_recall"], "val_small_class_f1": mv["small_class_f1"],
                     "test_mcc": mt["mcc"], "test_macro_f1": mt["macro_f1"], "test_accuracy": mt["accuracy"], "test_minority_recall": mt["minority_recall"],
                     "node_count": nodes, "depth": int(tree.get_depth()), "exported_kb": exported_kb(nodes),
                     "size_proxy_kb_upper_bound": nodes * EMBEDDED_BYTES_PER_NODE_EST / 1024,
                     "params": json.dumps(params, default=str)})
        best_so_far = -1.0
        for i, m in enumerate(hist, 1):
            best_so_far = max(best_so_far, m.get("cv_mcc_mean", 0.0))
            curves.append({"method": name, "repeat": rep, "eval": i, "cv_mcc": m.get("cv_mcc_mean"), "best_cv_mcc_so_far": best_so_far,
                           "embedded_size_kb": m.get("embedded_size_kb")})
        pd.DataFrame(rows).to_csv(raw_file, index=False); pd.DataFrame(curves).to_csv(cur_file, index=False)
        n_new += 1; elapsed = time.perf_counter() - t_start; remaining = total_runs - len(rows)
        log(f"repeat {rep+1}/{OPT_CMP_REPEATS} | {name:24s} val-MCC {mv['mcc']:.4f} | nodes {nodes:>7,} | search {search_s:5.0f}s refit {refit_s:4.0f}s | "
            f"~{elapsed/n_new*remaining/60:5.0f} min left")
print("\nfinished.", len(rows), "runs saved in", OUT)

## Reference points on the same scale
The saved models (`esp32_dt_final_results/<dataset>/metadata_*.json`: the searched model and the fixed-configuration model) are refitted on the same rows as the methods
(`OPT_CMP_REFIT_LIMIT`) and scored on the same validation and test rows, so they can be placed on the same scale.

In [ ]:
ref_file = OUT / "optimizer_comparison_reference.csv"
refs = []
if ref_file.exists():
    refs = pd.read_csv(ref_file).to_dict("records")
else:
    seen = set()
    for mf in sorted((Path("esp32_dt_final_results") / ACTIVE_DATASET).glob("metadata_*.json")):
        mj = json.loads(mf.read_text(encoding="utf-8")); key = json.dumps([mj["best_params"], mj["best_k"]], sort_keys=True, default=str)
        if key in seen: continue
        seen.add(key); prm, kk = dict(mj["best_params"]), min(int(mj["best_k"]), len(numeric_cols_check))
        Xr0, yr0 = stratified_sample(X_train, y_train, REFIT_N, SEED + 700)          # same rows as the methods' refit in repeat 0
        t1 = time.perf_counter(); pipe = build_pipeline(prm, kk, SEED + 700); pipe.fit(Xr0, yr0); fit_s = time.perf_counter() - t1
        mv = compute_enhanced_metrics(yv, pipe.predict(Xv), rare_classes, extremely_rare_classes); mt = compute_enhanced_metrics(yt, pipe.predict(Xt), rare_classes, extremely_rare_classes)
        nodes = int(pipe.named_steps["clf"].tree_.node_count)
        refs.append({"reference": f"{mf.stem.replace('metadata_', '')}: {prm.get('criterion')}, leaf={prm.get('min_samples_leaf')}, class_weight={prm.get('class_weight')}", "k": kk,
                     "val_mcc": mv["mcc"], "val_macro_f1": mv["macro_f1"], "val_minority_recall": mv["minority_recall"], "test_mcc": mt["mcc"], "test_macro_f1": mt["macro_f1"],
                     "node_count": nodes, "exported_kb": exported_kb(nodes, kk), "refit_rows": len(Xr0), "fit_seconds": fit_s, "params": json.dumps(prm, default=str)})
        print(f"reference {mf.stem}: val MCC {mv['mcc']:.4f} | test MCC {mt['mcc']:.4f} | nodes {nodes:,}", flush=True); del pipe
    pd.DataFrame(refs).to_csv(ref_file, index=False)
ref_df = pd.DataFrame(refs)
if len(ref_df): display(ref_df[["reference", "k", "val_mcc", "val_macro_f1", "test_mcc", "node_count", "exported_kb", "refit_rows"]].round(4))

## Summary table (median and IQR across repeats)

In [ ]:
raw = pd.read_csv(raw_file)
best_in_rep = raw.groupby("repeat")["val_mcc"].transform("max")
raw["deployable"] = (raw["node_count"] <= NODE_CAP) & (raw["val_mcc"] >= best_in_rep - MCC_TOL)
raw["rank_in_repeat"] = raw.groupby("repeat")["val_mcc"].rank(ascending=False)
raw.to_csv(raw_file, index=False)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - r) / d, (c + r) / d)

q25, q75 = (lambda s: s.quantile(.25)), (lambda s: s.quantile(.75))
g = raw.groupby("method")
summ = pd.DataFrame({
    "n_runs": g.size(), "evals_used": g["n_evals"].mean(),
    "val_mcc_median": g["val_mcc"].median(), "val_mcc_q25": g["val_mcc"].apply(q25), "val_mcc_q75": g["val_mcc"].apply(q75),
    "val_mcc_mean": g["val_mcc"].mean(), "val_mcc_std": g["val_mcc"].std(),
    "val_macro_f1_mean": g["val_macro_f1"].mean(), "val_minority_recall_mean": g["val_minority_recall"].mean(),
    "test_mcc_mean": g["test_mcc"].mean(), "test_macro_f1_mean": g["test_macro_f1"].mean(),
    "nodes_median": g["node_count"].median(), "nodes_q25": g["node_count"].apply(q25), "nodes_q75": g["node_count"].apply(q75),
    "exported_kb_median": g["exported_kb"].median(), "rescored_fitness_mean": g["rescored_fitness"].mean(),
    "mean_rank": g["rank_in_repeat"].mean(), "rank_std": g["rank_in_repeat"].std(),
    "deployable_rate": g["deployable"].mean(), "search_seconds_mean": g["search_seconds"].mean(),
})
ci = {m: wilson(int(gg["deployable"].sum()), len(gg)) for m, gg in g}
summ["deployable_ci_low"] = [ci[m][0] for m in summ.index]; summ["deployable_ci_high"] = [ci[m][1] for m in summ.index]
summ = summ.sort_values("val_mcc_median", ascending=False)
summ.to_csv(OUT / "optimizer_comparison_summary.csv")
print(f"deployable = nodes <= {NODE_CAP:,} AND val-MCC >= best run of that repeat - {MCC_TOL}")
display(summ[["n_runs", "val_mcc_median", "val_mcc_q25", "val_mcc_q75", "val_macro_f1_mean", "nodes_median", "exported_kb_median",
              "mean_rank", "deployable_rate", "deployable_ci_low", "deployable_ci_high"]].round(4))

## Paired statistics
Differences are computed within each repeat (same data, same seed). `mean_diff` = proposed − other. For MCC, macro-F1 and deployability, positive means the proposed method is better;
for `exported_kb`, negative means it is smaller. A bootstrap CI that includes 0 means there is no evidence of a difference. With 5 repeats the smallest possible Wilcoxon p-value is 0.0625.

In [ ]:
REF = "Full (S→D→P)"
rng_b = np.random.default_rng(0)
def paired(a, b, metric, higher_better=True):
    pa = raw[raw.method == a].set_index("repeat")[metric].astype(float); pb = raw[raw.method == b].set_index("repeat")[metric].astype(float)
    idx = pa.index.intersection(pb.index); d = (pa.loc[idx] - pb.loc[idx]).values
    if len(d) == 0: return None
    boots = rng_b.choice(d, size=(10000, len(d)), replace=True).mean(axis=1)
    try: p = _st.wilcoxon(d).pvalue if np.any(d != 0) and len(d) >= 5 else np.nan
    except Exception: p = np.nan
    sgn = 1 if higher_better else -1
    wins = ((sgn * d) > 0).sum() + 0.5 * (d == 0).sum()
    return {"proposed": a, "other": b, "metric": metric, "n_pairs": len(d), "mean_diff": d.mean(),
            "ci95_low": np.percentile(boots, 2.5), "ci95_high": np.percentile(boots, 97.5), "wilcoxon_p": p, "win_rate_proposed": wins / len(d)}

IMPROVED = [m for m in ("Full + restricted + MCC + no novelty bonus", "Halving-Silverback (aligned)") if m in METHODS]
st_rows = []
for ref_ in [REF] + IMPROVED:
    for other in [m for m in METHODS if m != ref_]:
        for metric, hb in [("val_mcc", True), ("val_macro_f1", True), ("val_minority_recall", True), ("exported_kb", False), ("deployable", True)]:
            r = paired(ref_, other, metric, hb)
            if r: st_rows.append(r)
stats_df = pd.DataFrame(st_rows); stats_df.to_csv(OUT / "optimizer_comparison_stats.csv", index=False)
show = stats_df[(stats_df.metric == "val_mcc") & (stats_df.proposed == REF)].copy()
show["verdict"] = np.where(show["n_pairs"] < 5, "too few repeats (need >= 5)",
                  np.where(show["ci95_low"] > 0, "proposed BETTER", np.where(show["ci95_high"] < 0, "proposed WORSE", "no clear difference")))
display(show[["other", "n_pairs", "mean_diff", "ci95_low", "ci95_high", "wilcoxon_p", "win_rate_proposed", "verdict"]].round(4))
for ref_ in IMPROVED:
    sh = stats_df[(stats_df.metric == "val_mcc") & (stats_df.proposed == ref_)].copy()
    sh["verdict"] = np.where(sh["n_pairs"] < 5, "too few repeats (need >= 5)", np.where(sh["ci95_low"] > 0, "proposed BETTER", np.where(sh["ci95_high"] < 0, "proposed WORSE", "no clear difference")))
    print(f"\n=== improved proposal: {ref_} (positive = it is better than 'other') ==="); display(sh[["other", "n_pairs", "mean_diff", "ci95_low", "ci95_high", "win_rate_proposed", "verdict"]].round(4))
print("\nSize (exported KB; negative = proposed smaller) and deployability:")
display(stats_df[stats_df.metric.isin(["exported_kb", "deployable"])][["other", "metric", "mean_diff", "ci95_low", "ci95_high", "win_rate_proposed"]].round(4))

## Figures

In [ ]:
order = list(summ.index)
_cols = [c for i, c in enumerate(plt.cm.tab10.colors) if i not in (0, 7)]      # keep blue for the proposed method, gray for default DT
palette = {m: _cols[i % len(_cols)] for i, m in enumerate([m for m in order if m not in (REF, "Default DT (no search)")])}
palette[REF] = "#1f77b4"; palette["Default DT (no search)"] = "#555555"

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
# (1) validation MCC distribution
ax = axes[0]
data = [raw[raw.method == m]["val_mcc"].values for m in order]
bp = ax.boxplot(data, vert=False, patch_artist=True, widths=.55, showfliers=False)
for patch, m in zip(bp["boxes"], order): patch.set_facecolor(palette[m]); patch.set_alpha(.35)
for i, (m, d) in enumerate(zip(order, data), 1):
    ax.scatter(d, np.full(len(d), i) + np.random.default_rng(i).uniform(-.15, .15, len(d)), s=14, color=palette[m], zorder=3)
ax.set_yticks(range(1, len(order) + 1), order, fontsize=7); ax.invert_yaxis(); ax.set_xlabel("validation MCC (each dot = one repeat)")
for _, rr in (ref_df.iterrows() if len(ref_df) else []):
    ax.axvline(rr["val_mcc"], color="black", ls="--", lw=1); ax.text(rr["val_mcc"], .02, " " + rr["reference"].split(":")[0][-8:], rotation=90, fontsize=6, va="bottom", transform=ax.get_xaxis_transform())
ax.set_title("validation MCC per repeat")
# (2) Pareto: size vs MCC
ax = axes[1]
for m in order:
    d = raw[raw.method == m]; ax.scatter(d["exported_kb"], d["val_mcc"], s=22, color=palette[m], alpha=.75, label=m)
ax.axvline(exported_kb(NODE_CAP), color="red", ls="--", lw=1); ax.text(exported_kb(NODE_CAP), ax.get_ylim()[0], " node cap", color="red", fontsize=7, va="bottom")
for _, rr in (ref_df.iterrows() if len(ref_df) else []):
    ax.scatter(rr["exported_kb"], rr["val_mcc"], marker="D", s=70, color="black", zorder=5); ax.annotate(" " + rr["reference"].split(":")[0][-8:], (rr["exported_kb"], rr["val_mcc"]), fontsize=6)
ax.set_xscale("log"); ax.set_xlabel("exported model size (KB, log)"); ax.set_ylabel("validation MCC"); ax.set_title("size–accuracy trade-off (one dot per run)")
ax.legend(fontsize=6, loc="lower right")
# (3) deployable success rate
ax = axes[2]
dr = summ["deployable_rate"].reindex(order); lo, hi = summ["deployable_ci_low"].reindex(order), summ["deployable_ci_high"].reindex(order)
ax.barh(order, dr, xerr=[dr - lo, hi - dr], color=[palette[m] for m in order], capsize=3, alpha=.8)
ax.invert_yaxis(); ax.set_xlim(0, 1.02); ax.set_yticks(range(len(order)), order, fontsize=7)
ax.set_xlabel("share of runs that are deployable (95% CI)"); ax.set_title("deployability success rate")
fig.suptitle(f"{ACTIVE_DATASET}: optimizer comparison, B = {B} evaluations, K = {BEST_K}, {raw.repeat.nunique()} repeats\n"
             f"deployable = at most {NODE_CAP:,} nodes AND validation MCC within {MCC_TOL} of the best run in that repeat", fontsize=10)
fig.savefig(FIGURES_DIR / "optimizer_comparison.png", dpi=300, bbox_inches="tight"); plt.show()

# convergence (bonus-free CV-MCC, best so far)
cur = pd.read_csv(cur_file)
fig, ax = plt.subplots(figsize=(7, 4))
for m in order:
    c = cur[cur.method == m]
    if c.empty: continue
    pivot = c.pivot_table(index="eval", columns="repeat", values="best_cv_mcc_so_far")
    med = pivot.median(axis=1); ax.plot(med.index, med.values, label=m, color=palette[m], lw=2.4 if m == REF else 1.3)
    ax.fill_between(med.index, pivot.quantile(.25, axis=1), pivot.quantile(.75, axis=1), color=palette[m], alpha=.08)
ax.set_xlabel("fitness evaluations"); ax.set_ylabel("best-so-far CV MCC (search sample)"); ax.legend(fontsize=7)
ax.set_title("search convergence: median and IQR over repeats (no novelty bonus)")
fig.savefig(FIGURES_DIR / "optimizer_comparison_convergence.png", dpi=300, bbox_inches="tight"); plt.show()